<a href="https://colab.research.google.com/github/Thanwarin/robot-webots/blob/tmp/emotion_detection_majorityvotin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Summary:

- Evaluates multiple models to compare performance.
- Mostly uses voice data as input.
- Results show that using multiple models gives lower accuracy than using a single model.
- Maintaining multiple models in practice could be time-consuming and costly.

Purpose: Explore the trade-offs between using multiple models versus a single model for voice-based emotion recognition.

In [ ]:
# Import Google Drive module to access files stored in Google Drive
from google.colab import drive

# Mount Google Drive to the Colab environment
# This allows you to read/write files from your Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install required libraries quietly (no detailed logs):
# - tensorflow: deep learning framework
# - keras: high-level neural network API
# - opencv-python: for image and video processing
# - mediapipe: for face and body landmark detection
!pip install tensorflow keras opencv-python mediapipe --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.


In [ ]:
# Install required libraries for facial and emotion recognition:
# - deepface: face recognition and analysis
# - fer: facial expression recognition
# - hsemotion: emotion detection
# - opencv-python-headless: image processing without GUI support
!pip install deepface fer hsemotion opencv-python-headless

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of facenet-pytorch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.1/891.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.0/297.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
# Download the FER-2013 facial emotion dataset CSV file
# from the GitLab repository and save it locally as 'fer2013.csv'
!wget -q https://gitlab.ecs.vuw.ac.nz/harisushehu/emotion-recognition-using-cnn/-/raw/master/dataset/fer2013.csv -O fer2013.csv

In [ ]:
# Install required libraries for deep learning and emotion recognition:
# - tensorflow: deep learning framework
# - deepface: face recognition and analysis
# - fer: facial expression recognition
# - hsemotion: emotion detection
# - opencv-python-headless: image processing without GUI support
!pip install tensorflow deepface fer hsemotion opencv-python-headless


In [ ]:
# Uninstall the 'fer' library from the environment without asking for confirmation
!pip uninstall -y fer

Found existing installation: fer 25.10.3
Uninstalling fer-25.10.3:
  Successfully uninstalled fer-25.10.3


In [ ]:
# Install specific versions of required libraries:
# - fer==22.4.0: facial expression recognition library
# - opencv-python-headless: image processing without GUI support
# - matplotlib: plotting and visualization library
!pip install fer==22.4.0 opencv-python-headless matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 15.2 MB/s eta 0:00:00


In [ ]:
# Install the DeepFace library for face recognition and analysis
!pip install deepface

In [ ]:
# =========================================
# Ensemble Prediction on FER2013
# Using your trained model + pre-trained libraries
# =========================================
# Install libraries (run once)
# !pip install tensorflow deepface fer hsemotion opencv-python-headless

# =========================================
# Imports
# =========================================
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import MobileNetV2, EfficientNetB3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input, Conv2D
from tensorflow.keras.models import Model
import numpy as np
import pandas as pd
import cv2
import os
from collections import Counter
from deepface import DeepFace
from fer import FER
from hsemotion.facial_emotions import HSEmotionRecognizer
from sklearn.model_selection import train_test_split

In [ ]:


# =========================================
# Config
# =========================================
IMG_SIZE = 48  # input size for FER2013
NUM_CLASSES = 7
SAVE_PATH = "/content/drive/MyDrive/ml_models"
FER_CSV_PATH = "/content/fer2013.csv"

# Map label index to emotion
emotion_map = {0:'angry',1:'disgust',2:'fear',3:'happy',4:'sad',5:'surprise',6:'neutral'}

# =========================================
# Load validation data
# =========================================
data = pd.read_csv(FER_CSV_PATH)
_, val_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data['emotion'])

def preprocess_img(pixel_sequence, img_size=IMG_SIZE):
    arr = np.array([int(p) for p in pixel_sequence.split()])
    img = arr.reshape(48,48).astype('uint8')
    img_resized = cv2.resize(img, (img_size,img_size))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
    img_rgb = img_rgb.astype('float32') / 255.0
    return img_rgb

X_val = np.array([preprocess_img(row['pixels']) for _, row in val_df.iterrows()])
y_val = np.array([row['emotion'] for _, row in val_df.iterrows()])

# =========================================
# Load your trained model (ver9)
# =========================================
trained_model_path = os.path.join(SAVE_PATH, "emotion_model_ver9.h5")
trained_model = load_model(trained_model_path)

In [ ]:


# =========================================
# Build MobileNetV2 & EfficientNetB3 from keras.applications
# as pre-trained models (for inference, no training)
# Convert grayscale input to 3 channels
# =========================================
def build_mobilenetv2():
    inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE,3), include_top=False, weights='imagenet')
    base.trainable = False
    x = base(inp, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    return model

def build_efficientnetb3():
    inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = EfficientNetB3(input_shape=(IMG_SIZE, IMG_SIZE,3), include_top=False, weights='imagenet')
    base.trainable = False
    x = base(inp, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    return model

mobilenet_model = build_mobilenetv2()
efficientnet_model = build_efficientnetb3()

# predict

In [ ]:
# =========================================
# Ensemble Prediction on FER2013
# =========================================
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import cv2
import os
from collections import Counter
from deepface import DeepFace
from fer import FER
from sklearn.model_selection import train_test_split

# =========================================
# Config
# =========================================
IMG_SIZE = 96  # Resize images for models
NUM_CLASSES = 7
SAVE_PATH = "/content/drive/MyDrive/ml_models"
FER_CSV_PATH = "/content/fer2013.csv"

# Pre-trained model paths
MODEL_PATHS = [
    os.path.join(SAVE_PATH, "emotion_model_ver9_final.h5"),
    os.path.join(SAVE_PATH, "emotion_model_mobilenetv2.h5"),
    os.path.join(SAVE_PATH, "emotion_model_efficientnetb3.h5")
]

# Map class index to emotion label
emotion_map = {0:'angry',1:'disgust',2:'fear',3:'happy',4:'sad',5:'surprise',6:'neutral'}

# =========================================
# Load FER2013 CSV and prepare validation set
# =========================================
data = pd.read_csv(FER_CSV_PATH)
_, val_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data['emotion'])

In [ ]:
# Preprocess function
def preprocess_img(pixel_sequence):
    arr = np.array([int(p) for p in pixel_sequence.split() if p.strip().isdigit()])
    if arr.size != 48*48:
        return None
    img = arr.reshape(48,48).astype('float32')
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img /= 255.0
    return img

X_val = []
y_val = []
for _, row in val_df.iterrows():
    img = preprocess_img(row['pixels'])
    if img is not None:
        X_val.append(img)
        y_val.append(row['emotion'])

X_val = np.array(X_val, dtype='float32')
y_val = np.array(y_val)

In [ ]:
# =========================================
# Load pre-trained TensorFlow/Keras models
# =========================================
models_list = []
for path in MODEL_PATHS:
    if os.path.exists(path):
        print("Loading model:", path)
        model = load_model(path)
        models_list.append(model)
    else:
        print("Model not found:", path)

# =========================================
# Initialize FER and DeepFace detectors
# =========================================
fer_detector = FER(mtcnn=True)

Loading model: /content/drive/MyDrive/ml_models/emotion_model_ver9_final.h5


Model not found: /content/drive/MyDrive/ml_models/emotion_model_mobilenetv2.h5
Model not found: /content/drive/MyDrive/ml_models/emotion_model_efficientnetb3.h5


In [ ]:
for i, img in enumerate(X_val):
    votes = []

    # 1. TensorFlow models
    img_input = np.expand_dims(img, axis=0)
    for model in models_list:
        pred = model.predict(img_input, verbose=0)
        pred_label = np.argmax(pred, axis=1)[0]
        votes.append(emotion_map[pred_label])

    # 2. FER
    fer_res = fer_detector.top_emotion((img*255).astype('uint8'))
    if fer_res[0] is not None:
        votes.append(fer_res[0])

    # 3. DeepFace
    try:
        df_res = DeepFace.analyze((img*255).astype('uint8'), actions=['emotion'], enforce_detection=False)
        votes.append(df_res['dominant_emotion'].lower())
    except:
        votes.append('neutral')  # fallback

    # Majority vote
    final_em = Counter(votes).most_common(1)[0][0]
    final_preds.append(final_em)

In [ ]:
# =========================================
# Ensemble Prediction on FER2013
# =========================================
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import cv2
import os
from collections import Counter
from deepface import DeepFace
from fer import FER
from sklearn.model_selection import train_test_split

# =========================================
# Config
# =========================================
IMG_SIZE = 96  # Resize images for models
NUM_CLASSES = 7
SAVE_PATH = "/content/drive/MyDrive/ml_models"
FER_CSV_PATH = "/content/fer2013.csv"

# Pre-trained model paths
MODEL_PATHS = [
    os.path.join(SAVE_PATH, "emotion_model_ver9_final.h5"),
    os.path.join(SAVE_PATH, "emotion_model_mobilenetv2.h5"),
    os.path.join(SAVE_PATH, "emotion_model_efficientnetb3.h5")
]

# Map class index to emotion label
emotion_map = {0:'angry',1:'disgust',2:'fear',3:'happy',4:'sad',5:'surprise',6:'neutral'}

# =========================================
# Load FER2013 CSV and prepare validation set
# =========================================
data = pd.read_csv(FER_CSV_PATH)
_, val_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data['emotion'])

# Preprocess function
def preprocess_img(pixel_sequence):
    arr = np.array([int(p) for p in pixel_sequence.split() if p.strip().isdigit()])
    if arr.size != 48*48:
        return None
    img = arr.reshape(48,48).astype('float32')
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img /= 255.0
    return img

X_val = []
y_val = []
for _, row in val_df.iterrows():
    img = preprocess_img(row['pixels'])
    if img is not None:
        X_val.append(img)
        y_val.append(row['emotion'])

X_val = np.array(X_val, dtype='float32')
y_val = np.array(y_val)

# =========================================
# Load pre-trained TensorFlow/Keras models
# =========================================
models_list = []
for path in MODEL_PATHS:
    if os.path.exists(path):
        print("Loading model:", path)
        model = load_model(path)
        models_list.append(model)
    else:
        print("Model not found:", path)

# =========================================
# Initialize FER and DeepFace detectors
# =========================================
fer_detector = FER(mtcnn=True)

# =========================================
# Predict and ensemble using majority vote
# =========================================
final_preds = []

for i, img in enumerate(X_val):
    votes = []

    # 1. TensorFlow models
    img_input = np.expand_dims(img, axis=0)
    for model in models_list:
        pred = model.predict(img_input, verbose=0)
        pred_label = np.argmax(pred, axis=1)[0]
        votes.append(emotion_map[pred_label])

    # 2. FER
    fer_res = fer_detector.top_emotion((img*255).astype('uint8'))
    if fer_res[0] is not None:
        votes.append(fer_res[0])

    # 3. DeepFace
    try:
        df_res = DeepFace.analyze((img*255).astype('uint8'), actions=['emotion'], enforce_detection=False)
        votes.append(df_res['dominant_emotion'].lower())
    except:
        votes.append('neutral')  # fallback

    # Majority vote
    final_em = Counter(votes).most_common(1)[0][0]
    final_preds.append(final_em)

# =========================================
# Compute accuracy
# =========================================
y_val_labels = [emotion_map[y] for y in y_val]
accuracy = np.mean([yt==yp for yt, yp in zip(y_val_labels, final_preds)])
print(f"Ensemble majority-vote accuracy on FER2013 val set: {accuracy*100:.2f}%")

# Example: show first 10 predictions
for i in range(10):
    print(f"True: {y_val_labels[i]}, Predicted: {final_preds[i]}")


7178

In [ ]:
# =========================================
# Ensemble Prediction on FER2013
# Using pre-trained models from Keras + your trained model + DeepFace + FER
# =========================================
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input, Conv2D
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import cv2
import os
from collections import Counter
from deepface import DeepFace
from fer import FER
from sklearn.model_selection import train_test_split

# =========================================
# Config
# =========================================
IMG_SIZE = 96  # Resize images for models
NUM_CLASSES = 7
SAVE_PATH = "/content/drive/MyDrive/ml_models"
FER_CSV_PATH = "/content/fer2013.csv"

# Path for the model you trained
TRAINED_MODEL_PATH = os.path.join(SAVE_PATH, "emotion_model_ver9_final.h5")

# Map class index to emotion label
emotion_map = {0:'angry',1:'disgust',2:'fear',3:'happy',4:'sad',5:'surprise',6:'neutral'}

# =========================================
# Load FER2013 CSV and prepare validation set
# =========================================
data = pd.read_csv(FER_CSV_PATH)
_, val_df = train_test_split(data, test_size=0.2, random_state=42, stratify=data['emotion'])

# Preprocess function
def preprocess_img(pixel_sequence):
    arr = np.array([int(p) for p in pixel_sequence.split() if p.strip().isdigit()])
    if arr.size != 48*48:
        return None
    img = arr.reshape(48,48).astype('float32')
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    img /= 255.0
    return img

X_val = []
y_val = []
for _, row in val_df.iterrows():
    img = preprocess_img(row['pixels'])
    if img is not None:
        X_val.append(img)
        y_val.append(row['emotion'])

X_val = np.array(X_val, dtype='float32')
y_val = np.array(y_val)

# =========================================
# Build pre-trained Keras models
# =========================================

def build_mobilenetv2():
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs, outputs)
    return model

def build_efficientnetb3():
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs
    )
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs, outputs)
    return model

mobilenet_model = build_mobilenetv2()
efficientnet_model = build_efficientnetb3()

# Load your trained model
trained_model = load_model(TRAINED_MODEL_PATH)

models_list = [trained_model, mobilenet_model, efficientnet_model]

# =========================================
# Initialize FER and DeepFace detectors
# =========================================
fer_detector = FER(mtcnn=True)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
# crate batch @ X_val
X_val_batch = X_val  # shape = (num_samples, IMG_SIZE, IMG_SIZE, 3)

all_preds = []
for model in models_list:
    preds = model.predict(X_val_batch, batch_size=32, verbose=1)  # batch predict
    preds_labels = np.argmax(preds, axis=1)
    all_preds.append(preds_labels)

# all_preds shape = [num_models, num_samples] -> transpose
all_preds = np.array(all_preds).T


fer_detector = FER(mtcnn=False)

df_res = DeepFace.analyze(
    (img*255).astype('uint8'),
    actions=['emotion'],
    enforce_detection=False,
    detector_backend='opencv'
)



def predict_emotion(idx_img):
    img = X_val[idx_img]
    votes = []

    # Keras models
    for pred_labels in all_preds[idx_img]:
        votes.append(emotion_map[pred_labels])

    # FER
    fer_res = fer_detector.top_emotion((img*255).astype('uint8'))
    if fer_res[0] is not None:
        votes.append(fer_res[0])

    # DeepFace
    try:
        df_res = DeepFace.analyze(
            (img*255).astype('uint8'),
            actions=['emotion'],
            enforce_detection=False,
            detector_backend='opencv'
        )
        votes.append(df_res['dominant_emotion'].lower())
    except:
        votes.append('neutral')

    return Counter(votes).most_common(1)[0][0]

with ThreadPoolExecutor(max_workers=4) as executor:
    final_preds = list(executor.map(predict_emotion, range(len(X_val))))


225/225 ━━━━━━━━━━━━━━━━━━━━ 53s 224ms/step
225/225 ━━━━━━━━━━━━━━━━━━━━ 52s 224ms/step
225/225 ━━━━━━━━━━━━━━━━━━━━ 185s 800ms/step


In [ ]:
# =========================================
# Compute accuracy
# =========================================
y_val_labels = [emotion_map[y] for y in y_val]
accuracy = np.mean([yt==yp for yt, yp in zip(y_val_labels, final_preds)])
print(f"Ensemble majority-vote accuracy on FER2013 val set: {accuracy*100:.2f}%")

Ensemble majority-vote accuracy on FER2013 val set: 59.21%


In [ ]:

# Example: show first 10 predictions
for i in range(10):
    print(f"True: {y_val_labels[i]}, Predicted: {final_preds[i]}")

True: happy, Predicted: happy
True: fear, Predicted: fear
True: happy, Predicted: happy
True: happy, Predicted: surprise
True: happy, Predicted: happy
True: fear, Predicted: happy
True: surprise, Predicted: surprise
True: surprise, Predicted: surprise
True: surprise, Predicted: surprise
True: happy, Predicted: happy


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ------------------------------------------------------
# 1) Load model (Model v9)
# ------------------------------------------------------
from tensorflow.keras.models import load_model
model_v9 = load_model("/content/drive/MyDrive/ml_models/emotion_model_ver9.h5")


# ------------------------------------------------------
# 2) emotion map
# ------------------------------------------------------
emotion_map = {
    0: "angry",
    1: "disgust",
    2: "fear",
    3: "happy",
    4: "sad",
    5: "surprise",
    6: "neutral"
}

# ------------------------------------------------------
# 3) Batch Predict
# ------------------------------------------------------
print("Predicting on validation set...")
pred_probs = model_v9.predict(X_val, batch_size=32, verbose=1)

# ans label index
pred_labels = np.argmax(pred_probs, axis=1)

# cover emotion string
pred_emotions = [emotion_map[p] for p in pred_labels]

# ------------------------------------------------------
# 4) Ground truth
# ------------------------------------------------------
y_true_labels = np.argmax(y_val, axis=1)
y_true_emotions = [emotion_map[y] for y in y_true_labels]

# ------------------------------------------------------
# 5) Accuracy + detailed metrics
# ------------------------------------------------------
acc = accuracy_score(y_true_labels, pred_labels)
print(f"\n Model v9 Accuracy: {acc*100:.2f}%")

# print("\nClassification Report:")
# print(classification_report(y_true_labels, pred_labels, target_names=list(emotion_map.values())))

# print("\nConfusion Matrix:")
# print(confusion_matrix(y_true_labels, pred_labels))


In [ ]:
y_true_labels = y_val.astype(int)
y_true_emotions = [emotion_map[y] for y in y_true_labels]

In [ ]:
print("\nClassification Report:")
print(classification_report(y_true_labels, pred_labels, target_names=list(emotion_map.values())))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_labels, pred_labels))


Classification Report:
              precision    recall  f1-score   support

       angry       0.55      0.52      0.54       991
     disgust       0.65      0.55      0.59       109
        fear       0.55      0.50      0.52      1024
       happy       0.83      0.82      0.82      1798
         sad       0.53      0.49      0.51      1216
    surprise       0.75      0.80      0.77       800
     neutral       0.54      0.65      0.59      1240

    accuracy                           0.64      7178
   macro avg       0.63      0.62      0.62      7178
weighted avg       0.64      0.64      0.64      7178


Confusion Matrix:
[[ 516   11   95   58  146   27  138]
 [  17   60    8    2   11    4    7]
 [ 118    7  510   38  142   92  117]
 [  58    2   44 1467   52   39  136]
 [ 126   10  142   81  597   18  242]
 [  23    1   50   30   19  637   40]
 [  74    2   73   89  167   33  802]]
